# xG-Spark: Expected Goals with Apache Spark
## Phase 2: Ingestion (Bronze Layer)

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent          # this notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from src.config import BRONZE_DIR
from src.spark_session import get_spark

spark = get_spark("xG-notebook")
spark


In [2]:
competitions = spark.read.parquet((BRONZE_DIR / "competitions").as_posix())
matches = spark.read.parquet((BRONZE_DIR / "matches").as_posix())
lineups = spark.read.parquet((BRONZE_DIR / "lineups").as_posix())
events = spark.read.parquet((BRONZE_DIR / "events").as_posix())

for name, df in [("competitions", competitions), ("matches", matches),
                 ("lineups", lineups), ("events", events)]:
    print(f"{name:<13} {df.count():>10,} rows  {len(df.columns):>3} columns")


competitions          80 rows   12 columns
matches            1,781 rows   18 columns
lineups            3,562 rows    4 columns
events         6,279,749 rows   45 columns


In [3]:
(events.groupBy("competition_id", "season_id").count()
    .join(competitions.select("competition_id", "season_id",
                              "competition_name", "season_name"),
          on=["competition_id", "season_id"])
    .orderBy(F.desc("count"))
    .show(truncate=False))


+--------------+---------+-------+----------------+-----------+
|competition_id|season_id|count  |competition_name|season_name|
+--------------+---------+-------+----------------+-----------+
|7             |27       |1358593|Ligue 1         |2015/2016  |
|12            |27       |1353739|Serie A         |2015/2016  |
|2             |27       |1313773|Premier League  |2015/2016  |
|11            |27       |1295354|La Liga         |2015/2016  |
|43            |106      |234637 |FIFA World Cup  |2022       |
|43            |3        |227825 |FIFA World Cup  |2018       |
|55            |43       |192664 |UEFA Euro       |2020       |
|55            |282      |187924 |UEFA Euro       |2024       |
|9             |27       |115240 |1. Bundesliga   |2015/2016  |
+--------------+---------+-------+----------------+-----------+



In [4]:
(events.groupBy(F.col("type.name").alias("event_type"))
    .count()
    .orderBy(F.desc("count"))
    .show(15))


+--------------+-------+
|    event_type|  count|
+--------------+-------+
|          Pass|1752850|
| Ball Receipt*|1641275|
|         Carry|1331430|
|      Pressure| 573997|
| Ball Recovery| 186672|
|          Duel| 142274|
|     Clearance|  83524|
|         Block|  69045|
|       Dribble|  60195|
|   Goal Keeper|  54308|
|Foul Committed|  53022|
|    Miscontrol|  51445|
|      Foul Won|  50446|
|  Dispossessed|  45599|
|          Shot|  44557|
+--------------+-------+
only showing top 15 rows



In [5]:
events.select("shot").printSchema()


root
 |-- shot: struct (nullable = true)
 |    |-- aerial_won: boolean (nullable = true)
 |    |-- body_part: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- deflected: boolean (nullable = true)
 |    |-- end_location: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- first_time: boolean (nullable = true)
 |    |-- follows_dribble: boolean (nullable = true)
 |    |-- freeze_frame: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- location: array (nullable = true)
 |    |    |    |    |-- element: double (containsNull = true)
 |    |    |    |-- player: struct (nullable = true)
 |    |    |    |    |-- id: long (nullable = true)
 |    |    |    |    |-- name: string (nullable = true)
 |    |    |    |-- position: struct (nullable = true)
 |    |    |    |    |-- id: long (nullable = true)
 |    |    |    |    |-- name: string (nul

In [6]:
(events.filter(F.col("type.name") == "Shot")
    .select("match_id", "minute",
            F.col("player.name").alias("player"),
            "location",
            F.col("shot.outcome.name").alias("outcome"),
            F.col("shot.statsbomb_xg").alias("statsbomb_xg"))
    .show(10, truncate=False))


+--------+------+--------------------------+-------------+-------+------------+
|match_id|minute|player                    |location     |outcome|statsbomb_xg|
+--------+------+--------------------------+-------------+-------+------------+
|3795108 |1     |Xherdan Shaqiri           |[91.9, 39.0] |Blocked|0.030168695 |
|3795108 |7     |Jordi Alba Ramos          |[96.5, 29.7] |Blocked|0.021315211 |
|3795108 |16    |Jorge Resurrección Merodio|[97.8, 25.9] |Off T  |0.047888376 |
|3795108 |24    |César Azpilicueta Tanco   |[112.0, 40.7]|Saved  |0.10035629  |
|3795108 |33    |Haris Seferović           |[99.7, 58.6] |Blocked|0.008757749 |
|3795108 |33    |Manuel Obafemi Akanji     |[113.2, 39.9]|Off T  |0.08995936  |
|3795108 |38    |Silvan Widmer             |[112.9, 34.3]|Off T  |0.06444093  |
|3795108 |46    |Daniel Olmo Carvajal      |[104.9, 30.2]|Saved  |0.08699033  |
|3795108 |47    |Manuel Obafemi Akanji     |[111.9, 38.8]|Wayward|0.11480155  |
|3795108 |49    |Jorge Resurrección Mero

In [7]:
print("Partitions in events DataFrame:", events.rdd.getNumPartitions())

# Look for "PartitionFilters" in the plan: Spark reads ONLY the Premier League folder
events.filter(F.col("competition_id") == 2).select("id").explain()


Partitions in events DataFrame: 23
== Physical Plan ==
*(1) Project [id#85]
+- *(1) ColumnarToRow
   +- FileScan parquet [id#85,competition_id#111,season_id#112] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/adity/Desktop/xg_spark/data/bronze/events], PartitionFilters: [isnotnull(competition_id#111), (competition_id#111 = 2)], PushedFilters: [], ReadSchema: struct<id:string>




## Phase 3 - Preprocessing (Silver Layer)

In [8]:
from src.config import SILVER_DIR

shots = spark.read.parquet((SILVER_DIR / "shots").as_posix())
print(f"{shots.count():,} shots, {len(shots.columns)} columns")
shots.printSchema()


43,924 shots, 36 columns
root
 |-- match_id: long (nullable = true)
 |-- shot_id: string (nullable = true)
 |-- competition_id: integer (nullable = true)
 |-- season_id: integer (nullable = true)
 |-- period: long (nullable = true)
 |-- minute: long (nullable = true)
 |-- second: long (nullable = true)
 |-- possession: long (nullable = true)
 |-- team_id: long (nullable = true)
 |-- team_name: string (nullable = true)
 |-- player_id: long (nullable = true)
 |-- player_name: string (nullable = true)
 |-- position: string (nullable = true)
 |-- play_pattern: string (nullable = true)
 |-- x: double (nullable = true)
 |-- y: double (nullable = true)
 |-- under_pressure: boolean (nullable = true)
 |-- shot_type: string (nullable = true)
 |-- body_part: string (nullable = true)
 |-- technique: string (nullable = true)
 |-- first_time: boolean (nullable = true)
 |-- one_on_one: boolean (nullable = true)
 |-- open_goal: boolean (nullable = true)
 |-- aerial_won: boolean (nullable = true)
 |-- 

In [9]:
(shots.select("match_date", "competition_name", "team_name", "player_name",
              "minute", "x", "y", "shot_type", "body_part", "outcome",
              "is_goal", "is_home")
    .show(5, truncate=False))


+----------+----------------+--------------+----------------------------+------+-----+----+---------+----------+-------+-------+-------+
|match_date|competition_name|team_name     |player_name                 |minute|x    |y   |shot_type|body_part |outcome|is_goal|is_home|
+----------+----------------+--------------+----------------------------+------+-----+----+---------+----------+-------+-------+-------+
|2024-06-23|UEFA Euro       |Switzerland   |Kwadwo Duah                 |73    |93.6 |51.6|Open Play|Right Foot|Saved  |0      |true   |
|2021-06-14|UEFA Euro       |Poland        |Karol Linetty               |50    |108.9|49.4|Open Play|Left Foot |Saved  |0      |true   |
|2016-04-20|La Liga         |Rayo Vallecano|Javier Guerra Rodríguez     |46    |115.3|40.9|Open Play|Right Foot|Post   |0      |false  |
|2024-07-10|UEFA Euro       |Netherlands   |Xavi Simons                 |76    |103.5|45.0|Open Play|Right Foot|Saved  |0      |true   |
|2015-08-29|La Liga         |Sporting Gij

In [10]:
null_counts = shots.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in shots.columns]
)
null_counts.show(vertical=True)


-RECORD 0------------------
 match_id          | 0     
 shot_id           | 0     
 competition_id    | 0     
 season_id         | 0     
 period            | 0     
 minute            | 0     
 second            | 0     
 possession        | 0     
 team_id           | 0     
 team_name         | 0     
 player_id         | 0     
 player_name       | 0     
 position          | 0     
 play_pattern      | 0     
 x                 | 0     
 y                 | 0     
 under_pressure    | 0     
 shot_type         | 0     
 body_part         | 0     
 technique         | 0     
 first_time        | 0     
 one_on_one        | 0     
 open_goal         | 0     
 aerial_won        | 0     
 follows_dribble   | 0     
 redirect          | 0     
 key_pass_id       | 12191 
 freeze_frame      | 0     
 statsbomb_xg      | 0     
 outcome           | 0     
 is_goal           | 0     
 match_date        | 0     
 competition_name  | 0     
 season_name       | 0     
 competition_stage |

In [11]:
shots.groupBy("outcome").count().orderBy(F.desc("count")).show()


+----------------+-----+
|         outcome|count|
+----------------+-----+
|           Off T|14682|
|         Blocked|11220|
|           Saved|10161|
|            Goal| 4158|
|         Wayward| 2575|
|            Post|  834|
|Saved Off Target|  158|
|   Saved to Post|  136|
+----------------+-----+



In [12]:
def conversion_table(df, column):
    """Shots, goals, conversion rate and average StatsBomb xG for each group."""
    return (df.groupBy(column)
            .agg(F.count("*").alias("shots"),
                 F.sum("is_goal").alias("goals"),
                 F.round(F.avg("is_goal"), 3).alias("conversion"),
                 F.round(F.avg("statsbomb_xg"), 3).alias("avg_statsbomb_xg"))
            .orderBy(F.desc("shots")))

conversion_table(shots, "body_part").show(truncate=False)


+----------+-----+-----+----------+----------------+
|body_part |shots|goals|conversion|avg_statsbomb_xg|
+----------+-----+-----+----------+----------------+
|Right Foot|22773|2120 |0.093     |0.088           |
|Left Foot |13749|1229 |0.089     |0.088           |
|Head      |7275 |773  |0.106     |0.108           |
|Other     |127  |36   |0.283     |0.298           |
+----------+-----+-----+----------+----------------+



In [13]:
conversion_table(shots, "play_pattern").show(truncate=False)


+--------------+-----+-----+----------+----------------+
|play_pattern  |shots|goals|conversion|avg_statsbomb_xg|
+--------------+-----+-----+----------+----------------+
|Regular Play  |14791|1442 |0.097     |0.094           |
|From Free Kick|9126 |826  |0.091     |0.082           |
|From Throw In |7872 |695  |0.088     |0.087           |
|From Corner   |7134 |598  |0.084     |0.09            |
|From Counter  |1978 |307  |0.155     |0.142           |
|From Goal Kick|1716 |163  |0.095     |0.095           |
|From Keeper   |748  |78   |0.104     |0.09            |
|From Kick Off |496  |37   |0.075     |0.084           |
|Other         |63   |12   |0.19      |0.213           |
+--------------+-----+-----+----------+----------------+



In [14]:
conversion_table(shots, "competition_name").show(truncate=False)


+----------------+-----+-----+----------+----------------+
|competition_name|shots|goals|conversion|avg_statsbomb_xg|
+----------------+-----+-----+----------+----------------+
|Serie A         |9877 |858  |0.087     |0.084           |
|Premier League  |9817 |914  |0.093     |0.092           |
|La Liga         |9071 |945  |0.104     |0.1             |
|Ligue 1         |8723 |852  |0.098     |0.092           |
|FIFA World Cup  |3068 |287  |0.094     |0.094           |
|UEFA Euro       |2538 |220  |0.087     |0.091           |
|1. Bundesliga   |830  |82   |0.099     |0.099           |
+----------------+-----+-----+----------+----------------+



## Phase 4 - Feature Engineering

In [15]:
features = spark.read.parquet((SILVER_DIR / "shot_features").as_posix())
print(f"{features.count():,} shots, {len(features.columns)} columns")

(features.select("player_name", "x", "y", "distance", "angle", "in_box",
                 "defenders_in_cone", "nearest_defender", "gk_distance",
                 "assist_type", "is_goal")
    .show(5, truncate=False))


43,924 shots, 43 columns
+---------------------+-----+----+------------------+-------------------+------+-----------------+------------------+------------------+-----------+-------+
|player_name          |x    |y   |distance          |angle              |in_box|defenders_in_cone|nearest_defender  |gk_distance       |assist_type|is_goal|
+---------------------+-----+----+------------------+-------------------+------+-----------------+------------------+------------------+-----------+-------+
|Lucas Pérez Martínez |104.3|56.4|22.703523955544878|0.24636794357429193|1     |0                |1.9416487838947676|19.670282153543198|Other Pass |0      |
|Rachid Ghezzal       |107.8|57.4|21.250882334623192|0.22041838523994128|1     |0                |3.2649655434629   |16.786303941010956|Other Pass |0      |
|Wahbi Khazri         |106.0|69.0|32.202484376209235|0.10925952291192709|0     |1                |2.23606797749979  |26.832815729997478|Other Pass |0      |
|Piotr Zieliński      |83.5 |44.2

In [16]:
numeric_features = ["distance", "log_distance", "angle", "in_box", "defenders_in_cone",
                    "nearest_defender", "gk_distance", "gk_off_line"]

features.select(numeric_features).summary("count", "min", "25%", "50%", "75%", "max", "mean").show(truncate=False)


+-------+------------------+-------------------+-------------------+------------------+------------------+------------------+------------------+------------------+
|summary|distance          |log_distance       |angle              |in_box            |defenders_in_cone |nearest_defender  |gk_distance       |gk_off_line       |
+-------+------------------+-------------------+-------------------+------------------+------------------+------------------+------------------+------------------+
|count  |43924             |43924              |43924              |43924             |43924             |43924             |43924             |43924             |
|min    |0.6324555320336713|0.49008534261440606|0.0                |0                 |0                 |0.0               |0.3605551275463981|0.0               |
|25%    |12.397580409095958|2.595074124167399  |0.25968348550471554|0                 |0                 |1.4317821063276304|9.062008607367359 |1.4000000000000057|
|50%    |19.0685

In [17]:
conversion_table(features, "assist_type").show(truncate=False)


+------------+-----+-----+----------+----------------+
|assist_type |shots|goals|conversion|avg_statsbomb_xg|
+------------+-----+-----+----------+----------------+
|Other Pass  |23512|1651 |0.07      |0.073           |
|No Assist   |12191|1135 |0.093     |0.088           |
|Cross       |6016 |823  |0.137     |0.135           |
|Through Ball|1416 |432  |0.305     |0.225           |
|Cutback     |789  |117  |0.148     |0.153           |
+------------+-----+-----+----------+----------------+



In [18]:
conversion_table(features, "defenders_in_cone").orderBy("defenders_in_cone").show(truncate=False)


+-----------------+-----+-----+----------+----------------+
|defenders_in_cone|shots|goals|conversion|avg_statsbomb_xg|
+-----------------+-----+-----+----------+----------------+
|0                |21274|2905 |0.137     |0.129           |
|1                |15101|897  |0.059     |0.06            |
|2                |4840 |225  |0.046     |0.052           |
|3                |1804 |81   |0.045     |0.046           |
|4                |670  |33   |0.049     |0.048           |
|5                |193  |13   |0.067     |0.056           |
|6                |33   |3    |0.091     |0.058           |
|7                |6    |0    |0.0       |0.074           |
|8                |3    |1    |0.333     |0.078           |
+-----------------+-----+-----+----------+----------------+



In [19]:
by_distance = features.withColumn("distance_band", (F.floor(F.col("distance") / 5) * 5).cast("int"))

conversion_table(by_distance.filter("distance < 40"), "distance_band").orderBy("distance_band").show(truncate=False)


+-------------+-----+-----+----------+----------------+
|distance_band|shots|goals|conversion|avg_statsbomb_xg|
+-------------+-----+-----+----------+----------------+
|0            |595  |329  |0.553     |0.528           |
|5            |6146 |1318 |0.214     |0.205           |
|10           |8591 |1123 |0.131     |0.125           |
|15           |8146 |734  |0.09      |0.091           |
|20           |8037 |364  |0.045     |0.045           |
|25           |7296 |205  |0.028     |0.028           |
|30           |3720 |60   |0.016     |0.018           |
|35           |995  |16   |0.016     |0.008           |
+-------------+-----+-----+----------+----------------+



In [20]:
by_angle = features.withColumn("angle_band", F.round(F.floor(F.col("angle") / 0.2) * 0.2, 1))

conversion_table(by_angle, "angle_band").orderBy("angle_band").show(truncate=False)


+----------+-----+-----+----------+----------------+
|angle_band|shots|goals|conversion|avg_statsbomb_xg|
+----------+-----+-----+----------+----------------+
|0.0       |2579 |72   |0.028     |0.01            |
|0.2       |24675|1142 |0.046     |0.047           |
|0.4       |7977 |1014 |0.127     |0.127           |
|0.6       |4383 |712  |0.162     |0.145           |
|0.8       |2322 |462  |0.199     |0.199           |
|1.0       |1100 |324  |0.295     |0.293           |
|1.2       |445  |167  |0.375     |0.381           |
|1.4       |197  |93   |0.472     |0.463           |
|1.6       |113  |72   |0.637     |0.573           |
|1.8       |52   |33   |0.635     |0.6             |
|2.0       |28   |19   |0.679     |0.676           |
|2.2       |27   |25   |0.926     |0.843           |
|2.4       |10   |7    |0.7       |0.818           |
|2.6       |10   |10   |1.0       |0.868           |
|2.8       |3    |3    |1.0       |0.765           |
|3.0       |3    |3    |1.0       |0.936      

In [21]:
for column in ["distance", "log_distance", "angle", "defenders_in_cone", "nearest_defender", "gk_distance"]:
    print(f"{column:<18} {features.stat.corr(column, 'is_goal'):+.3f}")


distance           -0.239
log_distance       -0.274
angle              +0.295
defenders_in_cone  -0.113
nearest_defender   -0.017
gk_distance        -0.247


## Phase 5 - Spark SQL Analytics

In [22]:
from src.analytics import QUERIES
from src.config import BRONZE_DIR, GOLD_DIR, SILVER_DIR

spark.read.parquet((SILVER_DIR / "shot_features").as_posix()).createOrReplaceTempView("shot_features")
spark.read.parquet((BRONZE_DIR / "matches").as_posix()).createOrReplaceTempView("matches")

for name in QUERIES:
    table = spark.read.parquet((GOLD_DIR / name).as_posix())
    table.createOrReplaceTempView(name)
    print(f"{name:<20} {table.count():>6,} rows  {len(table.columns):>3} columns")


player_finishing        212 rows    8 columns
team_attack             194 rows   11 columns
match_xg              1,781 rows   11 columns
shot_zones              170 rows    6 columns
minute_trend              7 rows    5 columns
distance_body_part        5 rows    4 columns


In [23]:
spark.sql("""
    SELECT player_name, shots, goals, xg, goals_minus_xg, conversion
    FROM player_finishing
    ORDER BY goals_minus_xg DESC
    LIMIT 10
""").show(truncate=False)


+---------------------------------+-----+-----+-----+--------------+----------+
|player_name                      |shots|goals|xg   |goals_minus_xg|conversion|
+---------------------------------+-----+-----+-----+--------------+----------+
|Luis Alberto Suárez Díaz         |150  |39   |25.65|13.35         |0.26      |
|Zlatan Ibrahimović               |142  |31   |18.0 |13.0          |0.218     |
|Gonzalo Gerardo Higuaín          |182  |33   |22.97|10.03         |0.181     |
|Gareth Frank Bale                |91   |19   |9.79 |9.21          |0.209     |
|Lionel Andrés Messi Cuccittini   |195  |27   |18.99|8.01          |0.138     |
|Karim Benzema                    |108  |27   |19.15|7.85          |0.25      |
|Sergio Leonel Agüero del Castillo|119  |22   |14.42|7.58          |0.185     |
|Hatem Ben Arfa                   |87   |12   |5.87 |6.13          |0.138     |
|Mohamed Salah                    |78   |15   |9.32 |5.68          |0.192     |
|Riyad Mahrez                     |81   

In [24]:
spark.sql("""
    SELECT player_name, shots, goals, xg, goals_minus_xg, conversion
    FROM player_finishing
    ORDER BY goals_minus_xg
    LIMIT 5
""").show(truncate=False)


+------------------------------+-----+-----+-----+--------------+----------+
|player_name                   |shots|goals|xg   |goals_minus_xg|conversion|
+------------------------------+-----+-----+-----+--------------+----------+
|Edin Džeko                    |86   |6    |11.9 |-5.9          |0.07      |
|Aleksandar Mitrović           |114  |10   |15.21|-5.21         |0.088     |
|Martin Braithwaite Christensen|126  |10   |14.93|-4.93         |0.079     |
|Luca Toni                     |58   |3    |7.46 |-4.46         |0.052     |
|Giampaolo Pazzini             |76   |3    |7.43 |-4.43         |0.039     |
+------------------------------+-----+-----+-----+--------------+----------+



In [25]:
spark.sql("""
    SELECT competition_name, season_name, team_name, matches,
           goals_per_match, xg_per_match
    FROM team_attack
    WHERE xg_rank = 1
    ORDER BY competition_name, season_name
""").show(truncate=False)


+----------------+-----------+-------------------+-------+---------------+------------+
|competition_name|season_name|team_name          |matches|goals_per_match|xg_per_match|
+----------------+-----------+-------------------+-------+---------------+------------+
|1. Bundesliga   |2015/2016  |Bayer Leverkusen   |34     |1.44           |1.4         |
|FIFA World Cup  |2018       |Brazil             |5      |1.6            |2.46        |
|FIFA World Cup  |2022       |Germany            |3      |1.67           |2.48        |
|La Liga         |2015/2016  |Barcelona          |38     |2.58           |2.01        |
|Ligue 1         |2015/2016  |Paris Saint-Germain|37     |2.51           |1.91        |
|Premier League  |2015/2016  |Arsenal            |38     |1.61           |1.67        |
|Serie A         |2015/2016  |AS Roma            |38     |2.08           |1.55        |
|UEFA Euro       |2020       |Spain              |6      |1.67           |2.23        |
|UEFA Euro       |2024       |Po

In [26]:
spark.sql("""
    SELECT CASE WHEN result = 'Draw' THEN 'Draw'
                WHEN result = xg_favourite THEN 'xG favourite won'
                ELSE 'xG favourite lost' END AS outcome,
           COUNT(*) AS matches,
           ROUND(COUNT(*) / (SELECT COUNT(*) FROM match_xg), 3) AS share
    FROM match_xg
    GROUP BY outcome
    ORDER BY matches DESC
""").show(truncate=False)


+-----------------+-------+-----+
|outcome          |matches|share|
+-----------------+-------+-----+
|xG favourite won |977    |0.549|
|Draw             |465    |0.261|
|xG favourite lost|339    |0.19 |
+-----------------+-------+-----+



In [27]:
spark.sql("""
    SELECT zone_x, zone_y, shots, goals, conversion, avg_xg
    FROM shot_zones
    WHERE shots >= 30
    ORDER BY conversion DESC
    LIMIT 8
""").show(truncate=False)


+------+------+-----+-----+----------+------+
|zone_x|zone_y|shots|goals|conversion|avg_xg|
+------+------+-----+-----+----------+------+
|115   |40    |498  |229  |0.46      |0.457 |
|115   |35    |554  |253  |0.457     |0.442 |
|110   |35    |2197 |504  |0.229     |0.209 |
|110   |40    |2195 |435  |0.198     |0.198 |
|115   |45    |437  |70   |0.16      |0.157 |
|115   |30    |468  |73   |0.156     |0.15  |
|105   |40    |1683 |248  |0.147     |0.13  |
|110   |45    |1321 |188  |0.142     |0.137 |
+------+------+-----+-----+----------+------+



In [28]:
spark.sql("SELECT * FROM minute_trend ORDER BY minute_band").show(truncate=False)


+-----------+-----+-----+----------+------+
|minute_band|shots|goals|conversion|avg_xg|
+-----------+-----+-----+----------+------+
|00-15      |5836 |539  |0.092     |0.089 |
|15-30      |6571 |590  |0.09      |0.088 |
|30-45+     |7440 |682  |0.092     |0.089 |
|45-60      |7188 |705  |0.098     |0.095 |
|60-75      |7182 |685  |0.095     |0.093 |
|75-90+     |9482 |942  |0.099     |0.095 |
|Extra Time |225  |15   |0.067     |0.088 |
+-----------+-----+-----+----------+------+



In [29]:
spark.sql("SELECT * FROM distance_body_part ORDER BY distance_band").show(truncate=False)


+-------------+----------+---------+-----+
|distance_band|right_foot|left_foot|head |
+-------------+----------+---------+-----+
|00-06        |0.564     |0.527    |0.324|
|06-12        |0.264     |0.26     |0.105|
|12-18        |0.14      |0.123    |0.034|
|18-24        |0.055     |0.053    |0.029|
|24+          |0.024     |0.026    |0.0  |
+-------------+----------+---------+-----+



In [30]:
spark.sql(QUERIES["match_xg"]).explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [match_id#3854L, _extract_competition_name#4379 AS competition_name#4352, _extract_season_name#4380 AS season_name#4353, _extract_home_team_name#4381 AS home_team#4354, _extract_away_team_name#4382 AS away_team#4355, home_score#3944L, away_score#3940L, round(home_xg#4360, 3) AS home_xg#4356, round(away_xg#4361, 3) AS away_xg#4357, CASE WHEN (home_score#3944L > away_score#3940L) THEN Home WHEN (home_score#3944L < away_score#3940L) THEN Away ELSE Draw END AS result#4358, CASE WHEN (home_xg#4360 > away_xg#4361) THEN Home ELSE Away END AS xg_favourite#4359]
   +- BroadcastHashJoin [match_id#3854L], [match_id#3950L], Inner, BuildRight, false
      :- HashAggregate(keys=[match_id#3854L], functions=[sum(CASE WHEN (is_home#3887 = 1) THEN statsbomb_xg#3880 ELSE 0.0 END), sum(CASE WHEN (is_home#3887 = 0) THEN statsbomb_xg#3880 ELSE 0.0 END)])
      :  +- Exchange hashpartitioning(match_id#3854L, 16), ENSURE_REQUIREMENTS, [plan_id